In [1]:
%load_ext autoreload
%autoreload 2

import mlflow
from catboost import CatBoostRegressor
from data_prep import load_data, clean_price_pre_meter, convert_bool_to_int, clean_nan
from feature_eng import features, cat_features, parse_address
from train import train_model, log_experiment

In [ ]:
path = "../data/kufar_ads.csv"
df = load_data(path)
df = clean_price_pre_meter(df)
df = convert_bool_to_int(df)
df = clean_nan(df)
df.info()

Сконвертировано колонок: ['has_balcony', 'is_first_floor', 'is_last_floor']
[num] area_living: заполнено 4026 пропусков значением 0.00
[num] area_kitchen: заполнено 7445 пропусков значением 0.00
[num] year_built: заполнено 134 пропусков значением 0.00
[cat] bathroom_type: заполнено 4493 пропусков значением 'не_указано'
[cat] balcony_type: заполнено 3920 пропусков значением 'не_указано'
<class 'pandas.core.frame.DataFrame'>
Index: 10119 entries, 0 to 10149
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ad_id                 10119 non-null  int64  
 1   subject               10119 non-null  object 
 2   price_usd             10119 non-null  int64  
 3   price_per_meter_usd   10119 non-null  float64
 4   rooms                 10119 non-null  int64  
 5   area_total            10119 non-null  float64
 6   area_living           10119 non-null  float64
 7   area_kitchen          10119 non-null  float

In [3]:
df[["street", "house_number"]] = df["address"].apply(parse_address)
features = ["rooms", "year_built", "has_balcony", "is_first_floor", "is_last_floor",
            "area_total", "area_living", "area_kitchen", "bathroom_type", "balcony_type", "condition","street"]
cat_features = ["bathroom_type", "balcony_type", "condition", "street"]
df = clean_nan(df)
print(features)

[cat] house_number: заполнено 1838 пропусков значением 'не_указано'
['rooms', 'year_built', 'has_balcony', 'is_first_floor', 'is_last_floor', 'area_total', 'area_living', 'area_kitchen', 'bathroom_type', 'balcony_type', 'condition', 'street']


In [4]:
X = df[features]
y = df["price_per_meter_usd"]

In [5]:
print(set(cat_features) - set(X.columns))

set()


In [6]:
X.head()

,rooms,year_built,has_balcony,is_first_floor,is_last_floor,area_total,area_living,area_kitchen,bathroom_type,balcony_type,condition,street
0,2,2021.0,1,0,0,42.1,38.5,0.0,Совмещенный,Лоджия,Вторичное,Леонида Левина ул
1,2,2003.0,1,0,0,70.0,36.0,12.0,Раздельный,Есть,Вторичное,Мазурова ул
2,1,2027.0,1,0,0,34.1,0.0,0.0,Совмещенный,Лоджия,Новое,площадь Старый Аэропорт
3,1,2027.0,1,0,0,34.1,0.0,0.0,Совмещенный,Лоджия,Новое,площадь Старый Аэропорт
4,1,2027.0,0,0,0,27.8,22.0,0.0,Совмещенный,не_указано,Новое,площадь Старый Аэропорт


In [7]:
y.head()

0    2922.00
1    2171.00
2    1772.93
3    1772.93
4    1837.08
Name: price_per_meter_usd, dtype: float64

In [10]:
mlflow.set_experiment("catboost-kufar-kv")
log_params = {
    "iterations": 900,
    "depth": 5,
    "learning_rate": 0.02,
}
unlog_params = {
    "random_state": 42,
    "verbose": False,
}
model_params = log_params | unlog_params
#holdout
mode = "cv"

with mlflow.start_run():

    model = CatBoostRegressor(**model_params, cat_features=cat_features)

    results, n_rows = train_model(
        model=model,
        X=X,
        y=y,
        mode=mode,
        test_size=0.2,
        random_state=42,
    )

    log_experiment(results, model, log_params, mode, features, path, n_rows)

    

Доступные метрики: ['rmse', 'nrmse', 'r2']
RMSE:  377.38
NRMSE: 18.35%
R2:    0.6200
dict_items([('iterations', 900), ('learning_rate', 0.02), ('depth', 5), ('loss_function', 'RMSE'), ('verbose', False), ('random_state', 42), ('cat_features', ['bathroom_type', 'balcony_type', 'condition', 'street'])])
